# Assignment 4: Categorical Data Handling

**Professor:** Subodh Kumar  
**Assignment:** 4  
**Marks:** 100

## Objective

Download a relevant dataset from **UCI Machine Learning Repository or Kaggle** and perform categorical data handling using Python.

This notebook is based on **Chapter 5 – Handling Categorical Data** from *Machine Learning with Python Cookbook* by O'Reilly.

## Dataset Selected

### UCI Adult / Census Income Dataset

The Adult dataset contains demographic and employment-related attributes such as:

- Workclass
- Education
- Marital status
- Occupation
- Relationship
- Race
- Sex
- Native country

The target is income category (`<=50K` or `>50K`).

This dataset is suitable for categorical data handling because it contains several nominal and ordinal categorical variables and missing categorical values represented by `?`.

## Main Categorical Data Handling Tasks

1. Identify categorical features
2. Count categorical values
3. Handle missing categorical values
4. Replace categorical values
5. Combine rare categories
6. Encode nominal categories
7. One-hot encode categories
8. Create dummy variables
9. Encode ordinal categories
10. Label encode a categorical target
11. Frequency encoding
12. Compare categorical representations
13. Check the final encoded dataset


## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import LabelEncoder

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## 2. Download and Load the UCI Dataset

The Adult dataset is downloaded directly from the UCI Machine Learning Repository.

The dataset does not contain a header row, so the column names are supplied manually.

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

columns = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
    "income"
]

df = pd.read_csv(
    url,
    names=columns,
    na_values="?",
    skipinitialspace=True
)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

df.head()

## 3. Understand the Dataset

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nDataset information:")
df.info()

## 4. Identify Categorical and Numerical Features

In [ ]:
categorical_columns = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_columns = df.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

print("\nNumerical columns:")
print(numerical_columns)

print("\nNumber of categorical features:", len(categorical_columns))
print("Number of numerical features:", len(numerical_columns))

## 5. Check Missing Categorical Values

The original Adult dataset represents missing categorical values with `?`. During loading, these values were converted to `NaN` using `na_values="?"`.

In [ ]:
missing_summary = df.isnull().sum()

missing_summary = (
    missing_summary[missing_summary > 0]
    .sort_values(ascending=False)
    .to_frame("Missing Values")
)

display(missing_summary)

print("Total missing values:", df.isnull().sum().sum())

## 6. Explore Categorical Values

`unique()` and `value_counts()` help us understand the categories present in each feature.

In [ ]:
for column in categorical_columns:
    print(f"\n{'=' * 60}")
    print(f"{column}")
    print(f"{'=' * 60}")
    print("Number of unique categories:", df[column].nunique(dropna=True))
    display(df[column].value_counts(dropna=False).head(15))

## 7. Visualise Categorical Variables

Bar charts are useful for comparing category frequencies.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

plot_columns = [
    "workclass",
    "education",
    "marital-status",
    "sex"
]

for ax, column in zip(axes.ravel(), plot_columns):
    order = df[column].value_counts().head(10).index
    sns.countplot(
        data=df,
        y=column,
        order=order,
        ax=ax
    )
    ax.set_title(f"Distribution of {column}")
    ax.set_xlabel("Count")

plt.tight_layout()
plt.show()

# 8. Handling Missing Categorical Values

### Mode Imputation

For categorical variables, the most frequent category (mode) is a simple way to fill missing observations.

Here, missing values are replaced with the mode of each categorical column.

In [ ]:
df_mode = df.copy()

for column in categorical_columns:
    if df_mode[column].isnull().any():
        mode_value = df_mode[column].mode()[0]
        df_mode[column] = df_mode[column].fillna(mode_value)
        print(f"{column}: missing values replaced with '{mode_value}'")

print("\nRemaining categorical missing values:")
print(df_mode[categorical_columns].isnull().sum().sum())

# 9. Handling Missing Categories Using a Separate Category

Another approach is to preserve the information that a value was missing.

Instead of replacing missing values with the mode, they are assigned to a new category called `Unknown`.

In [ ]:
df_unknown = df.copy()

for column in categorical_columns:
    df_unknown[column] = df_unknown[column].fillna("Unknown")

print("Remaining categorical missing values:",
      df_unknown[categorical_columns].isnull().sum().sum())

display(df_unknown[categorical_columns].head())

# 10. Replacing / Combining Categories

Categorical data sometimes contains categories that can be grouped into a broader meaningful category.

For example, education categories can be grouped into:

- School
- College
- Higher Education


In [ ]:
df_category = df_unknown.copy()

school_categories = [
    "Preschool",
    "1st-4th",
    "5th-6th",
    "7th-8th",
    "9th",
    "10th",
    "11th",
    "12th"
]

college_categories = [
    "HS-grad",
    "Some-college",
    "Assoc-voc",
    "Assoc-acdm"
]

higher_categories = [
    "Bachelors",
    "Masters",
    "Prof-school",
    "Doctorate"
]

def group_education(value):
    if value in school_categories:
        return "School"
    elif value in college_categories:
        return "College"
    elif value in higher_categories:
        return "Higher Education"
    else:
        return "Other"

df_category["education_group"] = df_category["education"].apply(
    group_education
)

display(
    df_category["education_group"]
    .value_counts()
)

## 10.1 Visualise the Grouped Categories

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(
    data=df_category,
    x="education_group",
    order=df_category["education_group"].value_counts().index
)

plt.title("Grouped Education Categories")
plt.xlabel("Education Group")
plt.ylabel("Number of Observations")
plt.xticks(rotation=20)
plt.show()

# 11. Handling Rare Categories

Rare categories can sometimes be combined into an `Other` category.

Here we group workclass categories that occur less than 1% of the observations.

In [ ]:
df_rare = df_unknown.copy()

column = "workclass"
frequency = df_rare[column].value_counts(normalize=True)

rare_categories = frequency[
    frequency < 0.01
].index

df_rare["workclass_grouped"] = df_rare[column].replace(
    rare_categories,
    "Other"
)

print("Rare categories grouped:")
print(list(rare_categories))

display(
    df_rare["workclass_grouped"]
    .value_counts()
)

# 12. One-Hot Encoding

### Encoding Nominal Categorical Data

Nominal categories do not have a natural order.

Examples:

- Sex
- Race
- Workclass
- Occupation

One-hot encoding creates a separate binary column for each category.

In [ ]:
onehot_data = df_unknown[
    ["sex", "race", "workclass"]
]

onehot_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoded_values = onehot_encoder.fit_transform(
    onehot_data
)

encoded_columns = onehot_encoder.get_feature_names_out(
    onehot_data.columns
)

df_onehot = pd.DataFrame(
    encoded_values,
    columns=encoded_columns,
    index=df_unknown.index
)

display(df_onehot.head())
print("Encoded shape:", df_onehot.shape)

# 13. Creating Dummy Variables with Pandas

Pandas provides `get_dummies()` as a convenient way to perform one-hot encoding.

In [ ]:
dummy_data = pd.get_dummies(
    df_unknown[["sex", "race"]],
    drop_first=True,
    dtype=int
)

display(dummy_data.head())

print("Dummy variable shape:", dummy_data.shape)

# 14. Ordinal Encoding

### Encoding Ordered Categories

Some categorical variables have a meaningful order.

Education is an example because education levels can be arranged from lower to higher levels.

We therefore define an explicit order before encoding.

In [ ]:
education_order = [
    "Preschool",
    "1st-4th",
    "5th-6th",
    "7th-8th",
    "9th",
    "10th",
    "11th",
    "12th",
    "HS-grad",
    "Some-college",
    "Assoc-voc",
    "Assoc-acdm",
    "Bachelors",
    "Masters",
    "Prof-school",
    "Doctorate"
]

df_ordinal = df_unknown.copy()

# Unknown values are placed after the known education levels
ordinal_categories = education_order + ["Unknown"]

ordinal_encoder = OrdinalEncoder(
    categories=[ordinal_categories],
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

df_ordinal["education_ordinal"] = ordinal_encoder.fit_transform(
    df_ordinal[["education"]]
)

display(
    df_ordinal[
        ["education", "education_ordinal"]
    ].head(15)
)

# 15. Label Encoding the Target

The target variable `income` contains two categories:

- `<=50K`
- `>50K`

For machine learning models, a categorical target can be converted into numerical labels.

In [ ]:
label_encoder = LabelEncoder()

income_encoded = label_encoder.fit_transform(
    df_unknown["income"]
)

df_target = df_unknown.copy()
df_target["income_encoded"] = income_encoded

print("Original classes:")
print(label_encoder.classes_)

print("\nEncoded values:")
display(
    df_target[
        ["income", "income_encoded"]
    ].head(10)
)

# 16. Frequency Encoding

Frequency encoding replaces each category with the frequency with which it appears in the dataset.

This can reduce the number of columns compared with one-hot encoding.

In [ ]:
df_frequency = df_unknown.copy()

frequency_map = (
    df_frequency["occupation"]
    .value_counts(normalize=True)
)

df_frequency["occupation_frequency"] = (
    df_frequency["occupation"]
    .map(frequency_map)
)

display(
    df_frequency[
        ["occupation", "occupation_frequency"]
    ].head(15)
)

# 17. Compare Encoding Methods

Different categorical encoding methods produce different representations.

### One-Hot Encoding
Creates one binary feature for each category.

### Ordinal Encoding
Assigns numbers according to a meaningful category order.

### Label Encoding
Converts target categories into numerical labels.

### Frequency Encoding
Replaces categories with their occurrence frequency.

In [ ]:
encoding_comparison = pd.DataFrame({
    "Original Sex": df_unknown["sex"].head(10).values,
    "Label Encoded Income": df_target["income_encoded"].head(10).values,
    "Ordinal Education": df_ordinal["education_ordinal"].head(10).values,
    "Occupation Frequency": df_frequency["occupation_frequency"].head(10).values
})

display(encoding_comparison)

# 18. Relationship Between Categorical Variables and Income

We can use a cross-tabulation to examine the relationship between sex and income.

In [ ]:
sex_income_table = pd.crosstab(
    df_unknown["sex"],
    df_unknown["income"],
    normalize="index"
) * 100

display(
    sex_income_table.round(2)
)

In [ ]:
sex_income_table.plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("Income Distribution by Sex")
plt.xlabel("Sex")
plt.ylabel("Percentage")
plt.xticks(rotation=0)
plt.legend(title="Income")
plt.tight_layout()
plt.show()

# 19. Final Encoded Dataset

For a machine learning workflow, categorical features can be combined with numerical features after encoding.

Below, selected categorical features are one-hot encoded and then combined with selected numerical features.

In [ ]:
selected_categorical = [
    "workclass",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex"
]

selected_numerical = [
    "age",
    "education-num",
    "capital-gain",
    "capital-loss",
    "hours-per-week"
]

encoded_categorical = pd.get_dummies(
    df_unknown[selected_categorical],
    drop_first=True,
    dtype=int
)

final_features = pd.concat(
    [
        df_unknown[selected_numerical],
        encoded_categorical
    ],
    axis=1
)

print("Final feature matrix shape:", final_features.shape)

display(final_features.head())

# 20. Verify the Final Dataset

The final encoded dataset should contain numerical values only, making it suitable as input to many machine learning algorithms.

In [ ]:
print("Any object columns remaining?")

object_columns = final_features.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print(object_columns)

print("\nFinal data types:")
display(final_features.dtypes.value_counts())

print("\nFinal shape:", final_features.shape)

# 21. Final Summary

In this assignment, categorical data handling was performed using the UCI Adult dataset.

### Operations Performed

| Operation | Completed |
|---|---|
| Load UCI dataset | ✓ |
| Identify categorical features | ✓ |
| Count unique categories | ✓ |
| Frequency analysis | ✓ |
| Missing categorical values | ✓ |
| Mode imputation | ✓ |
| Missing-as-category handling | ✓ |
| Category replacement/grouping | ✓ |
| Rare-category handling | ✓ |
| One-hot encoding | ✓ |
| Dummy variables | ✓ |
| Ordinal encoding | ✓ |
| Label encoding | ✓ |
| Frequency encoding | ✓ |
| Cross-tabulation | ✓ |
| Categorical visualisation | ✓ |
| Final numerical feature matrix | ✓ |

## Conclusion

Categorical variables cannot always be used directly by machine learning algorithms because many algorithms require numerical input. Categorical data therefore needs appropriate preprocessing.

**One-hot encoding** is useful for nominal categories because it does not impose an artificial order. **Ordinal encoding** is appropriate when categories have a meaningful natural order. **Label encoding** is useful for converting a categorical target into numerical labels. **Frequency encoding** provides a compact representation based on category frequency. Missing categories can be handled through mode imputation or by assigning a separate `Unknown` category, while rare categories can be grouped into `Other`.

The choice of encoding method should depend on the type of categorical variable and the requirements of the downstream machine learning algorithm.

# References

1. **UCI Machine Learning Repository – Adult Dataset**  
   https://archive.ics.uci.edu/dataset/2/adult

2. Dua, D. and Graff, C. (2019). *UCI Machine Learning Repository*. University of California, Irvine, School of Information and Computer Sciences.

3. *Machine Learning with Python Cookbook*, O'Reilly Media — Chapter 5: **Handling Categorical Data**.
